#### I aim to create a tags generator engine that takes data from e-commerce products name, description by combining fuzzy string matching, NLP semantics, classification, and content filtering into one pipeline.

___

#### Import Dependencies

In [1]:
import re
import unicodedata
import string
import spacy

##### Input Normalization

---

In [2]:
class TextNormalizer:
    def __init__(self, unicode_normalize: bool = True, strip_whitspace: bool = True, normalize_case: bool = True):
        self.unicode_normalize = unicode_normalize
        self.strip_whitspace = strip_whitspace
        self.normalize_case = normalize_case
        self.whitespace_pattern = re.compile( r"[\s\u200B-\u200D\uFEFF\u00A0\u202F]+")

    def normalize(self, text: str) -> str:
        if not isinstance(text, str):
            text = str(text)    #Convert non string into string
        if self.unicode_normalize:
            text = unicodedata.normalize('NFKC', text)
        if self.strip_whitspace:
            text = self.whitespace_pattern.sub(" ", text)
            text = text.strip()
        if self.normalize_case:
            text = text.casefold()
        return text

##### NLP and Tokenization

In [6]:
#noisy data generator

"""
Since there's not active dataset to train the Bert model, I've come up to turn normal product words 
into nuances, by using the following characteristics:

character deletion
character insertion
character substitution
character transposition
repeated characters
missing spaces
extra spaces
wrong spaces
phonetic mistakes
keyboard-neighbor mistakes
"""

"\nSince there's not active dataset to train the Bert model, I've come up to turn normal product words \ninto nuances, by using the following characteristics:\n\ncharacter deletion\ncharacter insertion\ncharacter substitution\ncharacter transposition\nrepeated characters\nmissing spaces\nextra spaces\nwrong spaces\nphonetic mistakes\nkeyboard-neighbor mistakes\n"

In [26]:
import random
def insert_character(text: str) -> str:
    index = random.randrange(len(text) + 1)
    char = random.choice(text.ascii_lowercase)

    return text[:index] + char + text[index:]

insert_character("Hello worlds")

AttributeError: 'str' object has no attribute 'ascii_lowercase'

In [69]:
import random


class NoiseGenerator:
    KEYBOARD_NEIGHBORS = {
        "q": "was",
        "w": "qase",
        "e": "wsdr",
        "r": "edft",
        "t": "rfgy",
        "y": "tugh",
        "u": "yhji",
        "i": "ujko",
        "o": "iklp",
        "p": "ol",
        "a": "qwsz",
        "s": "qweadzx",
        "d": "wersfxc",
        "f": "ertgdcv",
        "g": "rtyfhvb",
        "h": "tyugjbn",
        "j": "yuihkm",
        "k": "uiojln",
        "l": "opk",
        "z": "asx",
        "x": "zsdc",
        "c": "xdfv",
        "v": "cfgb",
        "b": "vghn",
        "n": "bhjm",
        "m": "njk",
    }

    def __init__(
        self,
        min_errors=1,
        max_errors=2,
        deletion_weight=0.35,
        transposition_weight=0.20,
        duplication_weight=0.10,
        keyboard_weight=0.35,
    ):
        self.min_errors = min_errors
        self.max_errors = max_errors

        self.error_types = [
            ("deletion", deletion_weight),
            ("transposition", transposition_weight),
            ("duplication", duplication_weight),
            ("keyboard", keyboard_weight),
        ]

    def delete_character(self, word):
        if len(word) <= 2:
            return word

        index = random.randrange(1, len(word) - 1)

        return word[:index] + word[index + 1:]

    def transpose_characters(self, word):
        if len(word) < 3:
            return word

        index = random.randrange(1, len(word) - 2)

        chars = list(word)

        chars[index], chars[index + 1] = (
            chars[index + 1],
            chars[index],
        )

        return "".join(chars)

    def duplicate_character(self, word):
        if len(word) < 2:
            return word

        index = random.randrange(1, len(word) - 1)

        return (
            word[:index]
            + word[index]
            + word[index:]
        )

    def keyboard_substitution(self, word):
        candidates = [
            i for i, char in enumerate(word)
            if char.lower() in self.KEYBOARD_NEIGHBORS
        ]

        if not candidates:
            return word

        index = random.choice(candidates)

        char = word[index].lower()

        replacement = random.choice(
            self.KEYBOARD_NEIGHBORS[char]
        )

        if word[index].isupper():
            replacement = replacement.upper()

        return (
            word[:index]
            + replacement
            + word[index + 1:]
        )

    def apply_error(self, word, error_type):
        if error_type == "deletion":
            return self.delete_character(word)

        if error_type == "transposition":
            return self.transpose_characters(word)

        if error_type == "duplication":
            return self.duplicate_character(word)

        if error_type == "keyboard":
            return self.keyboard_substitution(word)

        return word

    def generate(self, text):
        words = text.split()

        if not words:
            return text

        number_of_errors = random.randint(
            self.min_errors,
            min(self.max_errors, len(words))
        )

        selected_words = random.sample(
            range(len(words)),
            number_of_errors
        )

        for index in selected_words:

            error_type = random.choices(
                [x[0] for x in self.error_types],
                weights=[x[1] for x in self.error_types],
                k=1
            )[0]

            words[index] = self.apply_error(
                words[index],
                error_type
            )

        return " ".join(words)

In [70]:
noise = NoiseGenerator()

text = "wireless bluetooth headphones"

for _ in range(10):
    noisy = noise.generate(text)
    print(f"{text} -> {noisy}")

wireless bluetooth headphones -> wreless bluetooth headphons
wireless bluetooth headphones -> wireleess blietooth headphones
wireless bluetooth headphones -> wireless bluetooth headphonez
wireless bluetooth headphones -> wireless bluetoth headphones
wireless bluetooth headphones -> wirelsss bluetooth hearphones
wireless bluetooth headphones -> wireless bluetioth headhpones
wireless bluetooth headphones -> wireless bluetooth hadphones
wireless bluetooth headphones -> wireless bluettooth hewdphones
wireless bluetooth headphones -> wirekess bluetooth headphones
wireless bluetooth headphones -> wireless bluetooth hwadphones
